# S13 - Integración end-to-end y DataOps

Una sola ejecución recorre batch, streaming, modelo, observabilidad y BI, y deja un manifiesto que la hace reproducible y auditable.

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = (
    SparkSession.builder
    .appName("sesion13-integracion-dataops")
    .master("local[*]")
    .config("spark.jars.packages", "org.apache.spark:spark-sql-kafka-0-10_2.13:4.2.0")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
spark

:: loading settings :: url = jar:file:/usr/local/lib/python3.10/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /root/.ivy2.5.2/cache
The jars for the packages stored in: /root/.ivy2.5.2/jars
org.apache.spark#spark-sql-kafka-0-10_2.13 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-bcdfa03c-77bb-4a88-981c-6810b8f5a425;1.0
	confs: [default]


	found org.apache.spark#spark-sql-kafka-0-10_2.13;4.2.0 in central
	found org.apache.spark#spark-token-provider-kafka-0-10_2.13;4.2.0 in central
	found org.apache.kafka#kafka-clients;3.9.2 in central


	found at.yawk.lz4#lz4-java;1.11.0 in central
	found org.xerial.snappy#snappy-java;1.1.10.8 in central
	found org.slf4j#slf4j-api;2.0.17 in central
	found org.apache.hadoop#hadoop-client-runtime;3.5.0 in central
	found org.apache.hadoop#hadoop-client-api;3.5.0 in central


	found com.google.code.findbugs#jsr305;3.0.0 in central
	found org.scala-lang.modules#scala-parallel-collections_2.13;1.2.0 in central
	found org.apache.commons#commons-pool2;2.13.1 in central
:: resolution report :: resolve 764ms :: artifacts dl 29ms
	:: modules in use:
	at.yawk.lz4#lz4-java;1.11.0 from central in [default]
	com.google.code.findbugs#jsr305;3.0.0 from central in [default]
	org.apache.commons#commons-pool2;2.13.1 from central in [default]
	org.apache.hadoop#hadoop-client-api;3.5.0 from central in [default]
	org.apache.hadoop#hadoop-client-runtime;3.5.0 from central in [default]
	org.apache.kafka#kafka-clients;3.9.2 from central in [default]
	org.apache.spark#spark-sql-kafka-0-10_2.13;4.2.0 from central in [default]
	org.apache.spark#spark-token-provider-kafka-0-10_2.13;4.2.0 from central in [default]
	org.scala-lang.modules#scala-parallel-collections_2.13;1.2.0 from central in [default]
	org.slf4j#slf4j-api;2.0.17 from central in [default]
	org.xerial.snappy#snappy-java

26/11/06 03:42:54 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


## 1. Compuertas previas (*preflight*)

Si una dependencia no responde, la ejecución se detiene aquí con un mensaje claro, no cinco minutos después con un error de Spark.

In [2]:
import json, os, platform, time, urllib.request
from datetime import datetime, timezone

RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
INICIO = time.time()
manifiesto = {"runId": RUN_ID, "sparkVersion": spark.version, "python": platform.python_version(),
              "compuertas": {}, "batch": {}, "streaming": {}, "observabilidad": {}}

RUTA_EVENTOS_S9 = "../s09-observabilidad-pipeline/artifacts/atmos_eventos_observabilidad"
RUTA_MODELO_S10 = "../s10-series-tiempo-inferencia-streaming/artifacts/models/modelo_ce_lr"
TOPIC = "campo-electrico-eventos"

def http_ok(url):
    try:
        with urllib.request.urlopen(url, timeout=5) as r:
            return r.status == 200
    except Exception:
        return False

def kafka_filas(topic):
    return (spark.read.format("kafka")
            .option("kafka.bootstrap.servers", "kafka:9092")
            .option("subscribe", topic)
            .option("startingOffsets", "earliest").option("endingOffsets", "latest")
            .load().count())

compuertas = {
    "Parquet de S9 existe": os.path.isdir(RUTA_EVENTOS_S9),
    "Modelo de S10 existe": os.path.isdir(RUTA_MODELO_S10),
    "Pushgateway listo": http_ok("http://pushgateway:9091/-/ready"),
    "Prometheus listo": http_ok("http://prometheus:9090/-/ready"),
    "Grafana saludable": http_ok("http://grafana:3000/api/health"),
}
try:
    filas_topic = kafka_filas(TOPIC)
    compuertas["Kafka: topic " + TOPIC + " con datos"] = filas_topic > 0
except Exception as e:
    filas_topic = 0
    compuertas["Kafka: topic " + TOPIC + " con datos"] = False

for nombre, ok in compuertas.items():
    print(("OK    " if ok else "FALLA ") + nombre)
manifiesto["compuertas"] = compuertas
manifiesto["kafkaFilasTopic"] = filas_topic
fallidas = [n for n, ok in compuertas.items() if not ok]
assert not fallidas, "Compuertas que fallan: " + str(fallidas)

OK    Parquet de S9 existe
OK    Modelo de S10 existe
OK    Pushgateway listo
OK    Prometheus listo
OK    Grafana saludable
OK    Kafka: topic campo-electrico-eventos con datos


## 2. Batch: KPI de calidad y de flujo desde el Parquet de S9

In [3]:
eventos = spark.read.parquet(RUTA_EVENTOS_S9)
total = eventos.count()
nulos = eventos.select(
    F.sum(F.col("isValid").isNull().cast("int")).alias("isValid"),
    F.sum(F.col("latencyMs").isNull().cast("int")).alias("latencyMs"),
).first()

kpi = eventos.agg(
    F.avg(F.col("isValid").cast("int")).alias("ratioValidos"),
    F.avg("latencyMs").alias("latenciaPromedioMs"),
    F.expr("percentile_approx(latencyMs, 0.5)").alias("latenciaP50Ms"),
    F.expr("percentile_approx(latencyMs, 0.95)").alias("latenciaP95Ms"),
    F.max("latencyMs").alias("latenciaMaxMs"),
    F.avg((F.col("latencyMs") > 60000).cast("int")).alias("fraccionMayor1Min"),
).first()

compuertas_calidad = {
    "Hay filas": total > 0,
    "isValid sin nulos": (nulos["isValid"] or 0) == 0,
    "latencyMs sin nulos": (nulos["latencyMs"] or 0) == 0,
    "ratioValidos en [0, 1]": 0.0 <= kpi["ratioValidos"] <= 1.0,
    "Latencia tipica (p50) menor a 60 s": kpi["latenciaP50Ms"] < 60000,
}
for nombre, ok in compuertas_calidad.items():
    print(("OK    " if ok else "FALLA ") + nombre)
assert all(compuertas_calidad.values()), "La calidad del batch no pasa"

print(f"Eventos: {total:,}  |  ratio de validos: {kpi['ratioValidos']:.4f}")
print(f"Latencia: promedio {kpi['latenciaPromedioMs']:.0f} ms | p50 {kpi['latenciaP50Ms']} ms | p95 {kpi['latenciaP95Ms']} ms | max {kpi['latenciaMaxMs']} ms")
print(f"Eventos con mas de 1 minuto de latencia: {kpi['fraccionMayor1Min']:.1%}")
advertencias = []
if kpi["latenciaP95Ms"] > 300000:
    advertencias.append("p95 de latencia mayor a 5 minutos: el promedio no representa a la mayoria de los eventos")
for a in advertencias:
    print("ADVERTENCIA:", a)
manifiesto["batch"] = {"filas": total, "ratioValidos": kpi["ratioValidos"], "latenciaPromedioMs": kpi["latenciaPromedioMs"],
                       "latenciaP50Ms": kpi["latenciaP50Ms"], "latenciaP95Ms": kpi["latenciaP95Ms"],
                       "latenciaMaxMs": kpi["latenciaMaxMs"], "fraccionMayor1Min": kpi["fraccionMayor1Min"],
                       "calidad": compuertas_calidad, "advertencias": advertencias}

OK    Hay filas
OK    isValid sin nulos
OK    latencyMs sin nulos
OK    ratioValidos en [0, 1]
OK    Latencia tipica (p50) menor a 60 s
Eventos: 1,866  |  ratio de validos: 1.0000
Latencia: promedio 398793 ms | p50 11491 ms | p95 1488930 ms | max 1582393 ms
Eventos con mas de 1 minuto de latencia: 45.3%
ADVERTENCIA: p95 de latencia mayor a 5 minutos: el promedio no representa a la mayoria de los eventos


## 3. Streaming: el stream real de Kafka con el modelo de S10, durante una ventana fija de tiempo

In [4]:
from pyspark.ml import PipelineModel
from pyspark.sql.types import StructType, StringType, DoubleType, LongType

modelo = PipelineModel.load(RUTA_MODELO_S10)
esquema_evento = (
    StructType().add("tipoEvento", StringType()).add("estacionId", StringType())
    .add("CE", DoubleType()).add("CM", DoubleType()).add("TempOut", DoubleType())
    .add("OutHum", DoubleType()).add("WindSpeed", DoubleType()).add("Bar", DoubleType())
    .add("SolarRad", DoubleType()).add("UVIndex", DoubleType())
    .add("origen", StringType()).add("timestamp", LongType())
)
crudo = (spark.readStream.format("kafka")
         .option("kafka.bootstrap.servers", "kafka:9092").option("subscribe", TOPIC)
         .option("startingOffsets", "latest").option("failOnDataLoss", "false").load())
eventos_stream = (crudo.select(F.from_json(F.col("value").cast("string"), esquema_evento).alias("e")).select("e.*"))
predicciones = modelo.transform(eventos_stream)

acumulado = []
microlotes = []

def procesar_microlote(df, id_lote):
    filas = df.select("estacionId", "timestamp", "CE", "prediction").collect()
    microlotes.append({"batchId": id_lote, "filas": len(filas)})
    acumulado.extend((f["estacionId"], f["timestamp"], f["CE"], f["prediction"]) for f in filas)

DURACION_S = 45
consulta = predicciones.writeStream.foreachBatch(procesar_microlote).start()
consulta.awaitTermination(DURACION_S)
consulta.stop()

print("Micro-lotes con datos:", len([m for m in microlotes if m["filas"] > 0]))
print("Filas procesadas en", DURACION_S, "s:", len(acumulado))

netlib-blas: JNI_OnLoad: dlopen(libblas.so.3) failed: libblas.so.3: cannot open shared object file: No such file or directory


Micro-lotes con datos: 14
Filas procesadas en 45 s: 14


Exception in thread "stream execution thread for [id = 3907179a-5927-4190-baed-1af1c04a942f, runId = 64488168-eef9-4b35-aef1-615e0c904476]" java.lang.StackOverflowError
	at java.base/java.util.regex.Pattern$Branch.match(Pattern.java:4914)
	at java.base/java.util.regex.Pattern$GroupHead.match(Pattern.java:4969)
	at java.base/java.util.regex.Pattern$Loop.match(Pattern.java:5078)
	at java.base/java.util.regex.Pattern$GroupTail.match(Pattern.java:5000)
	at java.base/java.util.regex.Pattern$BranchConn.match(Pattern.java:4878)
	at java.base/java.util.regex.Pattern$CharProperty.match(Pattern.java:4110)
	at java.base/java.util.regex.Pattern$Branch.match(Pattern.java:4914)
	at java.base/java.util.regex.Pattern$GroupHead.match(Pattern.java:4969)
	at java.base/java.util.regex.Pattern$Loop.match(Pattern.java:5078)
	at java.base/java.util.regex.Pattern$GroupTail.match(Pattern.java:5000)
	at java.base/java.util.regex.Pattern$BranchConn.match(Pattern.java:4878)
	at java.base/java.util.regex.Pattern$B

va:5000)
	at java.base/java.util.regex.Pattern$BranchConn.match(Pattern.java:4878)
	at java.base/java.util.regex.Pattern$CharProperty.match(Pattern.java:4110)
	at java.base/java.util.regex.Pattern$Branch.match(Pattern.java:4914)
	at java.base/java.util.regex.Pattern$GroupHead.match(Pattern.java:4969)
	at java.base/java.util.regex.Pattern$Loop.match(Pattern.java:5078)
	at java.base/java.util.regex.Pattern$GroupTail.match(Pattern.java:5000)
	at java.base/java.util.regex.Pattern$BranchConn.match(Pattern.java:4878)
	at java.base/java.util.regex.Pattern$CharProperty.match(Pattern.java:4110)
	at java.base/java.util.regex.Pattern$Branch.match(Pattern.java:4914)
	at java.base/java.util.regex.Pattern$GroupHead.match(Pattern.java:4969)
	at java.base/java.util.regex.Pattern$Loop.match(Pattern.java:5078)
	at java.base/java.util.regex.Pattern$GroupTail.match(Pattern.java:5000)
	at java.base/java.util.regex.Pattern$BranchConn.match(Pattern.java:4878)
	at java.base/java.util.regex.Pattern$CharPropert

In [5]:
# MAE de produccion: la prediccion de una lectura contra el CE real de la lectura siguiente de la misma estacion
por_estacion = {}
for estacion, ts, ce, pred in sorted(acumulado, key=lambda x: (x[0], x[1])):
    por_estacion.setdefault(estacion, []).append((ce, pred))

errores = []
for lecturas in por_estacion.values():
    for (ce_actual, pred_actual), (ce_siguiente, _) in zip(lecturas, lecturas[1:]):
        errores.append(abs(pred_actual - ce_siguiente))

mae = sum(errores) / len(errores) if errores else None
print("Pares comparables:", len(errores), " | MAE en produccion:", None if mae is None else round(mae, 4))
assert acumulado, "El stream no entrego ninguna fila: esta corriendo el productor?"
manifiesto["streaming"] = {"duracionS": DURACION_S, "filas": len(acumulado), "microlotes": microlotes,
                           "paresComparables": len(errores), "maeProduccion": mae}

Pares comparables: 13  | MAE en produccion: 0.0423


## 4. Publicar los KPI de la ejecución en Pushgateway y confirmarlos en Prometheus

In [6]:
metricas = {
    "lambda26_int_batch_eventos_total": manifiesto["batch"]["filas"],
    "lambda26_int_batch_validos_ratio": manifiesto["batch"]["ratioValidos"],
    "lambda26_int_batch_latencia_p50_ms": manifiesto["batch"]["latenciaP50Ms"],
    "lambda26_int_batch_latencia_p95_ms": manifiesto["batch"]["latenciaP95Ms"],
    "lambda26_int_stream_filas_total": manifiesto["streaming"]["filas"],
    "lambda26_int_stream_mae": mae if mae is not None else 0,
    "lambda26_int_ultima_ejecucion_timestamp": int(time.time()),
}
cuerpo = "".join("# TYPE " + n + " gauge\n" + n + " " + str(v) + "\n" for n, v in metricas.items())
url = "http://pushgateway:9091/metrics/job/lambda26_integracion/instance/s13"
with urllib.request.urlopen(urllib.request.Request(url, data=cuerpo.encode(), method="PUT")) as r:
    print("Pushgateway respondio:", r.status)

def consultar(nombre):
    resultado = json.load(urllib.request.urlopen("http://prometheus:9090/api/v1/query?query=" + nombre))["data"]["result"]
    return resultado[0]["value"][1] if resultado else None

def coincide(obtenido, esperado):
    return obtenido is not None and abs(float(obtenido) - float(esperado)) <= 1e-6 * max(1.0, abs(float(esperado)))

confirmadas = {}
limite = time.time() + 90
while time.time() < limite:
    confirmadas = {n: consultar(n) for n in metricas}
    if all(coincide(confirmadas[n], v) for n, v in metricas.items()):
        break
    time.sleep(5)
for n, v in metricas.items():
    print(n + ":", confirmadas[n], "(publicado:", v, ")")
todas = all(coincide(confirmadas[n], v) for n, v in metricas.items())
manifiesto["observabilidad"] = {"metricas": confirmadas, "todasEnPrometheus": todas}
assert todas, "Prometheus no devolvio los valores recien publicados"

Pushgateway respondio: 200


lambda26_int_batch_eventos_total: 1866 (publicado: 1866 )
lambda26_int_batch_validos_ratio: 1 (publicado: 1.0 )
lambda26_int_batch_latencia_p50_ms: 11491 (publicado: 11491 )
lambda26_int_batch_latencia_p95_ms: 1488930 (publicado: 1488930 )
lambda26_int_stream_filas_total: 14 (publicado: 14 )
lambda26_int_stream_mae: 0.04234982151075391 (publicado: 0.04234982151075391 )
lambda26_int_ultima_ejecucion_timestamp: 1793936662 (publicado: 1793936662 )


## 5. Manifiesto de la ejecución

El manifiesto es lo que permite repetir, comparar y auditar una ejecución: qué versión corrió, con qué datos, qué compuertas pasó y qué resultados dio.

In [7]:
manifiesto["duracionTotalS"] = round(time.time() - INICIO, 1)
ruta_manifiesto = "./artifacts/manifiesto_" + RUN_ID + ".json"
with open(ruta_manifiesto, "w", encoding="utf-8") as f:
    json.dump(manifiesto, f, indent=2, ensure_ascii=False, default=str)
print("Manifiesto escrito en", ruta_manifiesto)
print(json.dumps({k: manifiesto[k] for k in ("runId", "sparkVersion", "duracionTotalS")}, indent=2))

Manifiesto escrito en ./artifacts/manifiesto_20261106T034302Z.json
{
  "runId": "20261106T034302Z",
  "sparkVersion": "4.2.0",
  "duracionTotalS": 90.0
}
